# Онлайн-рекомендации

## Шаг 1. Импорты и константы

В этом блоке подключаем библиотеки, которые понадобятся для загрузки данных, кодирования идентификаторов, построения sparse-матрицы и обучения ALS-модели.
Также здесь задаём базовые константы, чтобы не дублировать значения дальше по ноутбуку.

In [1]:
import numpy as np
import pandas as pd
import scipy
import sklearn.preprocessing

from implicit.als import AlternatingLeastSquares

import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# Точка разбиения событий на train и test.
# Всё, что раньше этой даты, пойдёт в обучение.
TRAIN_TEST_SPLIT_DATE = pd.to_datetime("2017-08-01").date()

# Сколько похожих объектов хотим хранить для каждого item.
# Запрашиваем на 1 больше, потому что similar_items возвращает сам объект тоже.
MAX_SIMILAR_ITEMS = 10

# Базовый размер выдачи для онлайн-рекомендаций.
TOP_K = 10

## Шаг 2. Загрузка данных

Загружаем таблицы с объектами и пользовательскими событиями.
Эти данные будут использоваться дальше для разбиения по времени, подготовки матрицы взаимодействий и построения похожих объектов.

In [2]:
# Загружаем данные об объектах и событиях пользователей.
items = pd.read_parquet("items.par")
events = pd.read_parquet("events.par")

# Быстрая проверка, что файлы прочитались корректно.
print(f"items: {items.shape}")
print(f"events: {events.shape}")

items: (43312, 19)
events: (11751086, 8)


## Шаг 3. Разбиение событий по времени

Для онлайн-рекомендаций важно сохранять хронологию.
Поэтому обучаемся на событиях, которые произошли раньше даты разбиения, а более поздние события оставляем для проверки и дальнейшей онлайн-логики.

In [3]:
# Делим события на train и test по фиксированной дате.
train_test_global_time_split_idx = events["started_at"] < TRAIN_TEST_SPLIT_DATE

events_train = events[train_test_global_time_split_idx].copy()
events_test = events[~train_test_global_time_split_idx].copy()

# Смотрим размер получившихся выборок.
print(f"events_train: {events_train.shape}")
print(f"events_test: {events_test.shape}")

events_train: (11326124, 8)
events_test: (424962, 8)


## Шаг 4. Подготовка данных для ALS

Перед обучением модели приводим названия колонок к единому виду и кодируем идентификаторы пользователей и объектов в последовательности целых чисел.
Это нужно для построения матрицы взаимодействий и корректной работы ALS.

In [4]:
# Приводим названия колонок к общему формату.
events_train = events_train.rename(columns={"book_id": "item_id"})
events_test = events_test.rename(columns={"book_id": "item_id"})
items = items.rename(columns={"book_id": "item_id"})

# Кодируем идентификаторы пользователей.
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# Кодируем идентификаторы объектов.
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["item_id"])

# Проверяем, что кодирование прошло корректно.
print(f"users encoded: {len(user_encoder.classes_)}")
print(f"items encoded: {len(item_encoder.classes_)}")
print(f"max train item_id_enc: {events_train['item_id_enc'].max()}")

users encoded: 430585
items encoded: 43312
max train item_id_enc: 43304


## Шаг 5. Матрица взаимодействий и обучение ALS

На этом шаге строим sparse-матрицу `user-item` по обучающим событиям и обучаем ALS-модель.
Именно эта модель дальше будет использоваться для поиска похожих объектов через `similar_items`.

In [9]:
# Размеры матрицы: число пользователей и число объектов.
n_rows = len(user_encoder.classes_)
n_cols = len(item_encoder.classes_)

# Строим sparse-матрицу взаимодействий user-item.
user_item_matrix_train = scipy.sparse.csr_matrix(
    (
        events_train["rating"],
        (events_train["user_id_enc"], events_train["item_id_enc"]),
    ),
    shape=(n_rows, n_cols),
    dtype=np.int8,
)

print(f"user_item_matrix_train shape: {user_item_matrix_train.shape}")
print(f"non-zero interactions: {user_item_matrix_train.nnz}")

# Обучаем ALS-модель на матрице взаимодействий.
als_model = AlternatingLeastSquares(
    factors=50,
    iterations=50,
    regularization=0.05,
    random_state=0,
)

# ограничиваем одним потоком
from threadpoolctl import threadpool_limits
with threadpool_limits(limits=1, user_api="blas"):
    als_model.fit(user_item_matrix_train)



user_item_matrix_train shape: (430585, 43312)
non-zero interactions: 11326124


  0%|          | 0/50 [00:00<?, ?it/s]

## Шаг 6. Подготовка `similar_items` (item2item)

Здесь строим набор похожих объектов через `als_model.similar_items`:
- для каждого `item_id` из train получаем `max_similar_items + 1` самых похожих (включая сам объект)
- разворачиваем таблицу в формат «одна строка = пара (item_1 -> item_2) с score»
- восстанавливаем исходные `item_id` и убираем self-pairs (`item_id_1 != item_id_2`)

После этого найдём объект, наиболее похожий на `item_id = 7126`.

In [10]:
# 1) Берем все энкодированные item_id, встречающиеся в train
train_item_ids_enc = events_train["item_id_enc"].unique()

# Метод similar_items возвращает и сам объект как наиболее похожий,
# поэтому запрашиваем на 1 больше и потом убираем self-pair.
max_similar_items = MAX_SIMILAR_ITEMS

similar_items_raw = als_model.similar_items(train_item_ids_enc, N=max_similar_items + 1)

sim_item_item_ids_enc = similar_items_raw[0]
sim_item_scores = similar_items_raw[1]

# 2) Сводим в DataFrame со списками похожих объектов и score
similar_items = pd.DataFrame({
    "item_id_enc": train_item_ids_enc,
    "sim_item_id_enc": sim_item_item_ids_enc.tolist(),
    "score": sim_item_scores.tolist(),
})

# 3) Разворачиваем списки в long-формат (одна строка = одна пара)
similar_items = similar_items.explode(["sim_item_id_enc", "score"], ignore_index=True)

# Приводим типы после explode
similar_items["sim_item_id_enc"] = similar_items["sim_item_id_enc"].astype("int")
similar_items["score"] = similar_items["score"].astype("float")

# 4) Восстанавливаем исходные item_id
similar_items["item_id_1"] = item_encoder.inverse_transform(similar_items["item_id_enc"])
similar_items["item_id_2"] = item_encoder.inverse_transform(similar_items["sim_item_id_enc"])

# Убираем тех. колонки с энкодингом
similar_items = similar_items.drop(columns=["item_id_enc", "sim_item_id_enc"])

# 5) Убираем пары, где объект похож сам на себя
similar_items = similar_items.query("item_id_1 != item_id_2")

# 6) Ответ на вопрос задания для item_id = 7126
item_to_check = 7126
subset = similar_items[similar_items["item_id_1"] == item_to_check]

if subset.empty:
    print(f"Не удалось найти item_id={item_to_check} среди item_id_1")
else:
    best = subset.sort_values("score", ascending=False).iloc[0]
    print(f"Наиболее похожий объект на {item_to_check}: {int(best['item_id_2'])}")
    # опционально: покажем top-5 для sanity check
    print(subset.sort_values("score", ascending=False).head(5)[["item_id_1", "item_id_2", "score"]].to_string(index=False))


Наиболее похожий объект на 7126: 7190
 item_id_1  item_id_2    score
      7126       7190 0.948713
      7126      24280 0.940992
      7126       1953 0.930133
      7126      58696 0.925048
      7126      38296 0.916306


In [11]:
# сохраняем
similar_items.to_parquet("similar_items.parquet") 

In [12]:
def print_sim_items(item_id, similar_items):
    '''
    Полезно убедиться, что полученный набор действительно содержит похожие данные. 
    Например, можно оценить глазами списки похожих объектов для каких-то уже известных. 
    '''
    item_columns_to_use = ["item_id", "author", "title", "genre_and_votes", "average_rating", "ratings_count"]
    
    item_id_1 = items.query("item_id == @item_id")[item_columns_to_use]
    display(item_id_1)
    
    si = similar_items.query("item_id_1 == @item_id")
    si = si.merge(items[item_columns_to_use].set_index("item_id"), left_on="item_id_2", right_index=True)
    display(si) 

print_sim_items(7144, similar_items) 
print_sim_items(16299, similar_items) 
print_sim_items(3, similar_items) 
print_sim_items(18135, similar_items) 
print_sim_items(17245, similar_items) 


,item_id,author,title,genre_and_votes,average_rating,ratings_count
1909078,7144,"Fyodor Dostoyevsky, David McDuff, Fyodor Dosto...",Crime and Punishment,"{'Classics': 15812, 'Fiction': 8028, 'Cultural...",4.19,390293


,score,item_id_1,item_id_2,author,title,genre_and_votes,average_rating,ratings_count
66760,0.964482,7144,12505,"Fyodor Dostoyevsky, Anna Brailovsky, Constance...",The Idiot,"{'Classics': 4036, 'Fiction': 2576}",4.18,76392
66761,0.953924,7144,12857,"Fyodor Dostoyevsky, Constance Garnett",The Gambler,"{'Classics': 946, 'Fiction': 729, 'Cultural-Ru...",3.88,22024
66762,0.952017,7144,67326,Fyodor Dostoyevsky,Poor Folk,"{'Classics': 320, 'Fiction': 235, 'Literature-...",3.73,4957
66763,0.946850,7144,5508624,Leo Tolstoy,Family Happiness,"{'Classics': 140, 'Fiction': 112, 'Cultural-Ru...",3.85,3337
66764,0.939767,7144,4934,"Fyodor Dostoyevsky, Fyodor Dostoyevsky, Richar...",The Brothers Karamazov,"{'Classics': 7496, 'Fiction': 5491, 'Cultural-...",4.31,158410
66765,0.938026,7144,17877,"Fyodor Dostoyevsky, Constance Garnett",The House of the Dead,"{'Classics': 533, 'Fiction': 441, 'Cultural-Ru...",4.04,8548
66766,0.937007,7144,929782,"Jack London, Andrew Sinclair",Martin Eden,"{'Classics': 435, 'Fiction': 405, 'Literature-...",4.39,13257
66767,0.936358,7144,28382,Nikolai Gogol,Diary of a Madman and Other Stories,"{'Classics': 284, 'Fiction': 243, 'Short Stori...",4.09,6241
66768,0.936324,7144,17690,"Franz Kafka, Max Brod, Willa Muir, Edwin Muir",The Trial,"{'Classics': 4607, 'Fiction': 4173, 'Literatur...",3.98,135862
66769,0.934545,7144,63038,Victor Hugo,The Man Who Laughs,"{'Classics': 352, 'Fiction': 176, 'Cultural-Fr...",4.22,5449


,item_id,author,title,genre_and_votes,average_rating,ratings_count
794321,16299,Agatha Christie,And Then There Were None,"{'Mystery': 12703, 'Classics': 6623, 'Fiction'...",4.23,429352


,score,item_id_1,item_id_2,author,title,genre_and_votes,average_rating,ratings_count
2388,0.858797,16299,16328,Agatha Christie,"The Murder of Roger Ackroyd (Hercule Poirot, #4)","{'Mystery': 5069, 'Fiction': 1765, 'Classics':...",4.20,74002
2389,0.817722,16299,16322,Agatha Christie,"The A.B.C. Murders (Hercule Poirot, #13)","{'Mystery': 3635, 'Fiction': 1085, 'Mystery-Cr...",3.98,51072
2390,0.811829,16299,16343,Agatha Christie,The Mysterious Affair at Styles (Hercule Poiro...,"{'Mystery': 5803, 'Fiction': 1826, 'Classics':...",3.98,142922
2391,0.794626,16299,16315,Agatha Christie,Crooked House,"{'Mystery': 1778, 'Fiction': 523, 'Mystery-Cri...",3.98,16312
2392,0.784923,16299,853510,Agatha Christie,"Murder on the Orient Express (Hercule Poirot, ...","{'Mystery': 9992, 'Classics': 4865, 'Fiction':...",4.16,25335
2393,0.781309,16299,131359,Agatha Christie,"Death on the Nile (Hercule Poirot, #17)","{'Mystery': 4103, 'Fiction': 1296, 'Mystery-Cr...",4.07,66646
2394,0.776504,16299,948072,"Charles Osborne, Agatha Christie",The Unexpected Guest,"{'Mystery': 311, 'Fiction': 88, 'Mystery-Crime...",3.99,3108
2395,0.774091,16299,121648,Agatha Christie,"Five Little Pigs (Hercule Poirot, #24)","{'Mystery': 1736, 'Fiction': 514, 'Mystery-Cri...",3.96,20208
2396,0.769657,16299,639787,Agatha Christie,"The Murder on the Links (Hercule Poirot, #2)","{'Mystery': 2341, 'Fiction': 672, 'Mystery-Cri...",3.80,19533
2397,0.761941,16299,16366,Agatha Christie,Endless Night,"{'Mystery': 1032, 'Fiction': 289, 'Mystery-Cri...",3.75,10155


,item_id,author,title,genre_and_votes,average_rating,ratings_count
1584855,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad...",4.45,4765497


,score,item_id_1,item_id_2,author,title,genre_and_votes,average_rating,ratings_count
10297,0.986768,3,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict...",4.38,1821802
10298,0.974957,3,5,"J.K. Rowling, Mary GrandPré",Harry Potter and the Prisoner of Azkaban (Harr...,"{'Fantasy': 49784, 'Young Adult': 15393, 'Fict...",4.53,1876252
10299,0.954404,3,6,"J.K. Rowling, Mary GrandPré",Harry Potter and the Goblet of Fire (Harry Pot...,"{'Fantasy': 48257, 'Young Adult': 15483, 'Fict...",4.53,1792561
10300,0.934245,3,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Order of the Phoenix (Har...,"{'Fantasy': 46485, 'Young Adult': 15194, 'Fict...",4.47,1766895
10301,0.922916,3,1,J.K. Rowling,Harry Potter and the Half-Blood Prince (Harry ...,"{'Fantasy': 46400, 'Young Adult': 15083, 'Fict...",4.54,1713866
10302,0.907906,3,136251,J.K. Rowling,Harry Potter and the Deathly Hallows (Harry Po...,"{'Fantasy': 46667, 'Young Adult': 15403, 'Fict...",4.62,1784684
10303,0.861319,3,8388506,"Bruno Nogueira, João Quadros","Tubo de Ensaio, Parte II","{'Humor': 4, 'Humor-Comedy': 1}",3.26,39
10304,0.861319,3,6379485,"Bruno Nogueira, João Quadros",Tubo de Ensaio,"{'Humor': 5, 'Humor-Comedy': 2}",3.27,44
10305,0.838425,3,7904207,Jim Henry,Antiquity Calais: Standing at Armageddon (The ...,None,4.61,16
10306,0.737692,3,8226034,Hans Scherfig,Frydenholm,"{'Historical-Historical Fiction': 3, 'Fiction'...",4.06,98


,item_id,author,title,genre_and_votes,average_rating,ratings_count
1592356,18135,"William Shakespeare, Paul Werstine, Barbara A....",Romeo and Juliet,"{'Classics': 26113, 'Plays': 9558, 'Fiction': ...",3.74,1656919


,score,item_id_1,item_id_2,author,title,genre_and_votes,average_rating,ratings_count
92951,0.920896,18135,8852,William Shakespeare,Macbeth,"{'Classics': 16116, 'Plays': 8310, 'Fiction': ...",3.88,502298
92952,0.889775,18135,1420,"William Shakespeare, Harold Bloom, Rex Gibson",Hamlet,"{'Classics': 17549, 'Plays': 8817, 'Fiction': ...",4.01,526122
92953,0.888849,18135,7728,"Sophocles, J.E. Thomas","Antigone (The Theban Plays, #3)","{'Classics': 3847, 'Plays': 2667, 'Drama': 897...",3.61,69075
92954,0.880328,18135,17250,"Arthur Miller, Christopher Bigsby",The Crucible,"{'Classics': 7902, 'Plays': 4768, 'Fiction': 2...",3.55,247565
92955,0.876483,18135,1622,"William Shakespeare, Paul Werstine, Barbara A....",A Midsummer Night's Dream,"{'Classics': 12032, 'Plays': 6438, 'Fiction': ...",3.94,340695
92956,0.869930,18135,12296,"Nathaniel Hawthorne, Thomas E. Connolly, Faust...",The Scarlet Letter,"{'Classics': 19456, 'Fiction': 6716}",3.37,515452
92957,0.866132,18135,13006,"William Shakespeare, Roma Gill",Julius Caesar,"{'Classics': 5864, 'Plays': 3637, 'Fiction': 1...",3.66,121890
92958,0.854553,18135,1554,"Sophocles, J.E. Thomas","Oedipus Rex (The Theban Plays, #1)","{'Classics': 4512, 'Plays': 3029, 'Drama': 100...",3.67,122126
92959,0.848532,18135,12996,William Shakespeare,Othello,"{'Classics': 8696, 'Plays': 5397, 'Fiction': 1...",3.88,242511
92960,0.836104,18135,2956,"Mark Twain, Guy Cardwell, John Seelye, Walter ...",The Adventures of Huckleberry Finn,"{'Classics': 20909, 'Fiction': 9269, 'Historic...",3.80,969291


,item_id,author,title,genre_and_votes,average_rating,ratings_count
1058909,17245,"Bram Stoker, Nina Auerbach, David J. Skal",Dracula,"{'Classics': 19603, 'Horror': 10601, 'Fiction'...",3.98,636895


,score,item_id_1,item_id_2,author,title,genre_and_votes,average_rating,ratings_count
23937,0.928806,17245,480204,"Gaston Leroux, Alexander Teixeira de Mattos",The Phantom of the Opera,"{'Classics': 7010, 'Fiction': 2103, 'Horror': ...",3.97,144859
23938,0.900328,17245,51496,"Robert Louis Stevenson, Vladimir Nabokov, Merv...",The Strange Case of Dr. Jekyll and Mr. Hyde,"{'Classics': 12342, 'Fiction': 4037, 'Horror':...",3.79,229898
23939,0.898929,17245,93261,Washington Irving,The Legend of Sleepy Hollow,"{'Classics': 2594, 'Horror': 1182, 'Fiction': ...",3.74,26776
23940,0.897704,17245,295,Robert Louis Stevenson,Treasure Island,"{'Classics': 11249, 'Fiction': 4405, 'Adventur...",3.82,274424
23941,0.896445,17245,2623,"Charles Dickens, Marisa Sestino",Great Expectations,"{'Classics': 19645, 'Fiction': 6662, 'Literatu...",3.75,468462
23942,0.895972,17245,18254,"Charles Dickens, Philip Horne, Gerald Dickens",Oliver Twist,"{'Classics': 11450, 'Fiction': 3656, 'Historic...",3.85,235560
23943,0.886892,17245,7190,Alexandre Dumas,"The Three Musketeers (The D'Artagnan Romances,...","{'Classics': 9823, 'Fiction': 3256, 'Historica...",4.06,198892
23944,0.881903,17245,24213,"Lewis Carroll, John Tenniel, Martin Gardner",Alice's Adventures in Wonderland & Through the...,"{'Classics': 11568, 'Fantasy': 6184, 'Fiction'...",4.06,344482
23945,0.878387,17245,2932,"Daniel Defoe, Virginia Woolf",Robinson Crusoe,"{'Classics': 7725, 'Fiction': 3305, 'Adventure...",3.66,181415
23946,0.870206,17245,1953,"Charles Dickens, Richard Maxwell",A Tale of Two Cities,"{'Classics': 20021, 'Fiction': 6969, 'Historic...",3.82,646983
